# MLflow Tracking and Error Analysis

Track model experiments and investigate incorrect predictions. Use validation metrics for model selection and keep final test metrics separate.

### Install Packages

In [1]:
%pip install mlflow

     ---------------------------------------- 0.0/50.2 kB ? eta -:--:--
     -------------------------------- ------- 41.0/50.2 kB 2.0 MB/s eta 0:00:01
     -------------------------------------- 50.2/50.2 kB 645.0 kB/s eta 0:00:00
     ---------------------------------------- 0.0/51.1 kB ? eta -:--:--
     -------------------------------- ------- 41.0/51.1 kB ? eta -:--:--
     -------------------------------------- 51.1/51.1 kB 523.4 kB/s eta 0:00:00
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
     ---------------------------------------- 0.0/44.3 kB ? eta -:--:--
     ---------------------------------------- 44.3/44.3 kB 2.3 MB/s eta 0:00:00
  Using cached protobuf-7.36.2-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached sqlparse-0.6.0-py3-none-any.whl.metadata (6.0 kB)
     ---------------------------------------- 0.0/115.0 kB ? eta -:--:--
     -------------------------------------- 115.0/115.0 kB 6.5 MB/s eta 0:00:00
  Using cached blinker-1.9.0-py3-n


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Track Baseline Experiments

In [18]:
# Install Libraries
import mlflow
import pandas as pd
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from mlflow import MlflowClient
from sklearn.metrics import classification_report

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

### Track Baseline Expereiments

In [ ]:
# Load and clean the dataset
data = pd.read_csv("../Dataset/cleaned_data/cleaned_montra_data.csv")

LABELS = ["negative", "neutral", "positive"]

data["sentiment_label"] = (data["sentiment_label"].astype(str).str.lower().str.strip())

data = data[data["heavy_text"].notna()
    & data["heavy_text"].astype(str).str.strip().ne("")
    & data["sentiment_label"].isin(LABELS)].copy()

if data.empty:
    raise ValueError("No valid reviews remain after filtering.")

# Split into training, validation and test sets
train_df, temp_df = train_test_split(data, test_size=0.40,
    random_state=42, stratify=data["sentiment_label"])

valid_df, test_df = train_test_split(temp_df, test_size=0.50,
    random_state=42, stratify=temp_df["sentiment_label"])

X_train = train_df["heavy_text"]
y_train = train_df["sentiment_label"]

X_valid = valid_df["heavy_text"]
y_valid = valid_df["sentiment_label"]

# Define baseline models
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000,class_weight="balanced"), 
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1, class_weight="balanced"),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(
        class_weight="balanced")}

OUT = Path("../Model/baseline")
OUT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = Path("../Results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Store the MLflow database and model artifacts in Results
MLFLOW_DB = RESULTS_DIR / "mlflow.db"
MODEL_DIR =Path("../Model/mlruns")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Configure MLflow tracking
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB.resolve().as_posix()}")

# Create a new experiment with artifacts in Results/mlruns
client = MlflowClient()
experiment_name = "Montra_Guest_Sentiment_Results"

experiment = client.get_experiment_by_name(experiment_name)

if experiment is None:
    client.create_experiment(
        name=experiment_name,
        artifact_location=MODEL_DIR.resolve().as_uri()
    )

mlflow.set_experiment(experiment_name)

print("Everything will be saved under:", RESULTS_DIR.resolve())
print("Database:", MLFLOW_DB.resolve())
print("Model artifacts:",MODEL_DIR.resolve())

# #=================================================================================
# # Get the project Results folder
# results_folder = Path.cwd().parent / "Results"

# # Create folders for MLflow data and model artifacts
# results_folder.mkdir(parents=True, exist_ok=True)

# mlruns_folder = results_folder / "mlruns"
# mlruns_folder.mkdir(parents=True, exist_ok=True)

# # Save experiment tracking information in SQLite
# mlflow_db = results_folder / "mlflow.db"

# mlflow.set_tracking_uri( f"sqlite:///{mlflow_db.resolve().as_posix()}")

# # Set the experiment
# mlflow.set_experiment("Montra_Guest_Sentiment")

# print("Results folder:", results_folder.resolve())
# print("MLflow database:", mlflow_db.resolve())
# print("Model artifacts folder:", mlruns_folder.resolve())

results = []

# Train, evaluate and log each model
for name, classifier in classifiers.items():

    with mlflow.start_run(run_name=name):

        pipeline = Pipeline([("tfidf", TfidfVectorizer(
                max_features=20000, ngram_range=(1, 2))),
            ("model", classifier)])

        # Train model
        pipeline.fit(X_train, y_train)

        # Predict validation sentiment
        pred = pipeline.predict(X_valid)

        # Calculate metrics
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_valid, pred, labels=LABELS, average="macro", zero_division=0)

        accuracy = accuracy_score(y_valid, pred)

        # Log parameters
        mlflow.log_param("model_name", name)
        mlflow.log_param("max_features", 20000)
        mlflow.log_param("ngram_range", "1,2")
        mlflow.log_param("class_labels", ",".join(LABELS))

        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("macro_precision", precision)
        mlflow.log_metric("macro_recall", recall)
        mlflow.log_metric("macro_f1", f1)

        # Log fitted pipeline
        mlflow.sklearn.log_model(sk_model=pipeline, name="pipeline", 
                                skops_trusted_types=["sklearn.tree._tree.Tree"])

        # Save results for comparison
        results.append({
            "Model": name,
            "Accuracy": accuracy,
            "Macro Precision": precision,
            "Macro Recall": recall,
            "Macro F1": f1})

# Compare models
results_data = (pd.DataFrame(results).sort_values("Macro F1", ascending=False)
    .round(4).reset_index(drop=True))


results_data.to_csv(
    RESULTS_DIR / "baseline_model_comparison.csv",
    index=False)

display(results_data)


Everything will be saved under: C:\Users\User\Documents\AMDARI_IMS\MONTRA_HOSPITALITY_PJ\Results
Database: C:\Users\User\Documents\AMDARI_IMS\MONTRA_HOSPITALITY_PJ\Results\mlflow.db
Model artifacts: C:\Users\User\Documents\AMDARI_IMS\MONTRA_HOSPITALITY_PJ\Model\mlruns


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,Random Forest,0.9397,0.9428,0.9400,0.9412
1,Logistic Regression,0.9393,0.9432,0.9390,0.9408
2,Linear SVM,0.9369,0.9413,0.9365,0.9385
3,Naive Bayes,0.9319,0.9378,0.9308,0.9336


In [20]:
mlflow.search_runs(
    experiment_names=["Montra_Guest_Sentiment"],
    order_by=["metrics.macro_f1 DESC"]
)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.macro_recall,metrics.macro_precision,metrics.accuracy,metrics.macro_f1,params.max_features,params.ngram_range,params.model_name,params.class_labels,tags.mlflow.user,tags.mlflow.source.type,tags.mlflow.source.name,tags.mlflow.runName
0,bbd0cf507b15476d98ee881b1de2f847,1,FINISHED,file:///c:/Users/User/Documents/AMDARI_IMS/MON...,2026-10-26 10:02:24.204000+00:00,2026-10-26 10:04:26.100000+00:00,0.940037,0.942751,0.939714,0.941213,20000,"1,2",Random Forest,"negative,neutral,positive",User,NOTEBOOK,mlflow_track_&_error_analysis.ipynb,Random Forest
1,5c161b8897814a77aaa91986fff15f85,1,FINISHED,file:///c:/Users/User/Documents/AMDARI_IMS/MON...,2026-10-26 09:52:39.709000+00:00,2026-10-26 09:54:59.673000+00:00,0.940037,0.942751,0.939714,0.941213,20000,"1,2",Random Forest,"negative,neutral,positive",User,NOTEBOOK,mlflow_track_&_error_analysis.ipynb,Random Forest
2,7f817120bc704c1b999c91594b4ca323,1,FINISHED,file:///c:/Users/User/Documents/AMDARI_IMS/MON...,2026-10-26 10:01:42.962000+00:00,2026-10-26 10:02:24.163000+00:00,0.939016,0.943204,0.939286,0.940788,20000,"1,2",Logistic Regression,"negative,neutral,positive",User,NOTEBOOK,mlflow_track_&_error_analysis.ipynb,Logistic Regression
3,3b7e12ba90d641e188fb228c1b2ddd4d,1,FINISHED,file:///c:/Users/User/Documents/AMDARI_IMS/MON...,2026-10-26 09:51:48.512000+00:00,2026-10-26 09:52:39.663000+00:00,0.939016,0.943204,0.939286,0.940788,20000,"1,2",Logistic Regression,"negative,neutral,positive",User,NOTEBOOK,mlflow_track_&_error_analysis.ipynb,Logistic Regression
4,2e0523694535449fba7fcb8176233506,1,FINISHED,file:///c:/Users/User/Documents/AMDARI_IMS/MON...,2026-10-26 10:05:03.368000+00:00,2026-10-26 10:05:39.604000+00:00,0.936457,0.941349,0.936857,0.938480,20000,"1,2",Linear SVM,"negative,neutral,positive",User,NOTEBOOK,mlflow_track_&_error_analysis.ipynb,Linear SVM
5,fb31874009504e2fb7ca5ac51fbab121,1,FINISHED,file:///c:/Users/User/Documents/AMDARI_IMS/MON...,2026-10-26 09:55:36.293000+00:00,2026-10-26 09:56:15.285000+00:00,0.936457,0.941349,0.936857,0.938480,20000,"1,2",Linear SVM,"negative,neutral,positive",User,NOTEBOOK,mlflow_track_&_error_analysis.ipynb,Linear SVM
6,94e3e68a3c0248f4a8729cf66105d807,1,FINISHED,file:///c:/Users/User/Documents/AMDARI_IMS/MON...,2026-10-26 10:04:26.223000+00:00,2026-10-26 10:05:03.263000+00:00,0.930790,0.937838,0.931857,0.933616,20000,"1,2",Naive Bayes,"negative,neutral,positive",User,NOTEBOOK,mlflow_track_&_error_analysis.ipynb,Naive Bayes
7,9b8f997f9ad7420fb7e37aae399285e4,1,FINISHED,file:///c:/Users/User/Documents/AMDARI_IMS/MON...,2026-10-26 09:54:59.708000+00:00,2026-10-26 09:55:36.240000+00:00,0.930790,0.937838,0.931857,0.933616,20000,"1,2",Naive Bayes,"negative,neutral,positive",User,NOTEBOOK,mlflow_track_&_error_analysis.ipynb,Naive Bayes


### Inspect Baseline Errors

In [21]:
# Load baseline test predictions
pred_file = Path("../Dataset/modelled_data/baseline_test_predictions.csv")

if pred_file.exists():

    errors_data = pd.read_csv(pred_file)

    # Check required columns
    required_cols = ["sentiment_label", "predicted_sentiment"]

    if not all(col in errors_data.columns for col in required_cols):
        raise ValueError("The prediction file is missing required columns.")

    # Select incorrectly classified reviews
    errors = errors_data[
        errors_data["sentiment_label"] != errors_data["predicted_sentiment"]].copy()

    print("Total test reviews:", len(errors_data))
    print("Misclassified reviews:", len(errors))

    # Display selected columns
    columns = [
        "review_id", "review_text", "Heavy_text",
        "sentiment_label", "predicted_sentiment", "hotel_name"]

    display_cols = [col for col in columns if col in errors.columns]

    display(errors[display_cols].head(30))

    # Compare actual and predicted sentiment for errors
    print("\nActual vs predicted sentiment for misclassified reviews:")

    display(pd.crosstab(
        errors["sentiment_label"],
        errors["predicted_sentiment"],
        rownames=["Actual sentiment"],
        colnames=["Predicted sentiment"]))

else:
    print("Prediction file not found.")
    print("Run notebook for baseline_sentiment first to create baseline_test_predictions.csv.")

Total test reviews: 7000
Misclassified reviews: 443


,review_id,review_text,sentiment_label,predicted_sentiment,hotel_name
3,24020,The fitness facilities left a lot to be desire...,neutral,negative,Nottingham Park Hotel
28,6908,We were disappointed that the neighborhood was...,neutral,negative,Liverpool Harbour Hotel
30,26252,I'd describe the night staff as acceptable — n...,positive,neutral,Glasgow Metropolitan
50,28838,The pricing stood out as well worth the cost; ...,positive,neutral,The Birmingham Grand
91,16131,"This hotel was a fine, if unremarkable, choice...",positive,neutral,The London Boutique
108,29361,I'd describe the spa as fine for our needs — w...,positive,neutral,Glasgow Skyline Hotel
126,3356,"This hotel was a fine, if unremarkable, choice...",negative,neutral,Liverpool Harbour Hotel
152,12336,"Unfortunately, the hotel staff was indifferent...",neutral,negative,Leicester Harbour Hotel
159,31155,"Overall the spa was basic but functional, abou...",positive,neutral,The London Boutique
173,6608,"Sadly, the business center turned out to be po...",neutral,negative,The Leeds Grand



Actual vs predicted sentiment for misclassified reviews:


Predicted sentiment,negative,neutral,positive
Actual sentiment,,,
negative,0,142,1
neutral,68,0,82
positive,0,150,0


## 4. Start the MLflow UI

Run the below command in a separate VS Code terminal, not in a Python cell:

**mlflow server --host 127.0.0.1 --port 5000**

 Then open http://127.0.0.1:5000 in your browser.